In [1]:
## We used the same file to train on GRPO
## We modified the parameters for every configuration
## For this specific configuration we ran it on 1000 samples parameters but later faced Memory constraints

# 🚀 GRPO Training — Qwen2.5 Math
Loads SFT LoRA weights, then fine-tunes further with GRPO on GSM8K.

**Pipeline:**
```
Stage 1 ✅  SFT on NuminaMath-CoT  →  stable reasoning generation
Stage 2 🔜  GRPO on GSM8K          →  RL with correctness reward  ← YOU ARE HERE
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📦 Step 1: Install Dependencies

In [ ]:
%%capture
!pip install -q torch==2.3.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.46.3
!pip install -q trl==0.15.1
!pip install -q peft==0.13.2
!pip install -q bitsandbytes==0.43.3
!pip install -q accelerate==1.1.1
!pip install -q datasets==3.1.0
!pip install -q triton==2.3.0
!pip install -q sentencepiece
!pip install hf_transfer
print('✅ All packages installed!')

## 🖥️ Step 2: Verify Environment

In [ ]:
import torch, re, json, os, warnings
warnings.filterwarnings('ignore')

print(f'🖥️  Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'🎮 GPU:   {gpu.name}')
    print(f'💾 VRAM: {gpu.total_memory / 1e9:.1f} GB')
    print(f'🔧 CUDA: {torch.version.cuda}')

🖥️  Device: CUDA
🎮 GPU:   Tesla T4
💾 VRAM: 15.6 GB
🔧 CUDA: 12.1


## ⚙️ Step 3: Configuration
**Edit these paths and hyperparameters before running.**

In [ ]:
# ── Paths ─────────────────────────────────────────────────────
MODEL_ID        = "Qwen/Qwen2.5-1.5B"   # base model on HuggingFace
ADAPTER_PATH    = "/content/drive/MyDrive/RL Assignement/sft"               # ← folder with your SFT LoRA adapter files
GRPO_OUTPUT_DIR = "/content/drive/MyDrive/RL Assignement/qwen-math-grpo"
SAVE_DIR        = "/content/drive/MyDrive/RL Assignement/grpo_checkpoints"  # checkpoints + final model saved here

# ── Hyperparameters ───────────────────────────────────────────
GRPO_SAMPLES    = 1000   # number of GSM8K training samples
GRPO_EPOCHS     = 1
GRPO_LR         = 5e-6   # keep lower than your SFT LR
GRPO_BATCH_SIZE = 2
GRPO_GRAD_ACCUM = 8
MAX_NEW_TOKENS  = 256
NUM_GENERATIONS = 2      # responses sampled per prompt (higher = better signal, more VRAM)
KL_BETA         = 0.01   # KL penalty — higher keeps model closer to SFT policy

print('⚙️  Config loaded!')

⚙️  Config loaded!


In [ ]:
# Utility: Evaluate model accuracy on 50 samples
def evaluate_model_accuracy(model, tokenizer, dataset, num_samples=50):
    model.eval()
    correct = 0
    total = 0
    for i, example in enumerate(dataset.select(range(num_samples))):
        prompt = example['prompt']
        answer = example['answer']
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=32)
        completion = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Use the same extract_number logic as in reward
        pred = extract_number(completion)
        gold = extract_number(answer)
        if pred and gold and pred == gold:
            correct += 1
        total += 1
    accuracy = 100 * correct / total if total > 0 else 0
    print(f"\n🔎 Accuracy on {total} samples: {accuracy:.2f}%")
    return accuracy


## 📥 Step 4: Load SFT Model + Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig

print('📥 Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
base_model.config.use_cache = False

print('📥 Loading SFT LoRA adapters...')
peft_config = PeftConfig.from_pretrained(ADAPTER_PATH)
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    config=peft_config,
    is_trainable=True,
    local_files_only=True,
)

print('📥 Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
vram_gb   = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f'\n✅ Model ready!')
print(f'   Trainable params: {trainable/1e6:.1f}M')
print(f'   VRAM used:        {vram_gb:.2f} GB')

📥 Loading base model...


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

📥 Loading SFT LoRA adapters...
📥 Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


✅ Model ready!
   Trainable params: 18.5M
   VRAM used:        3.16 GB


## 📊 Step 5: Load & Format Dataset

In [ ]:
from datasets import load_dataset

print('📦 Loading GSM8K dataset...')
dataset = load_dataset('openai/gsm8k', 'main', split='train')
dataset = dataset.select(range(GRPO_SAMPLES))

def format_prompt(example):
    return {
        'prompt': (
            'You are a math reasoning assistant. '
            'Solve the problem step by step, then give your final answer as a number.\n\n'
            f'Problem: {example["question"]}\n\n'
            'Solution:'
        ),
        'answer': example['answer'].split('####')[-1].strip(),
    }

dataset = dataset.map(format_prompt)
print(f'✅ Dataset ready: {len(dataset)} samples')
print(f'\nExample prompt:\n{dataset[0]["prompt"]}')
print(f'Answer: {dataset[0]["answer"]}')

📦 Loading GSM8K dataset...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

✅ Dataset ready: 3000 samples

Example prompt:
You are a math reasoning assistant. Solve the problem step by step, then give your final answer as a number.

Problem: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Solution:
Answer: 72


## 🏆 Step 6: Reward Functions

In [ ]:
import re

def extract_number(text):
    """Extract the last number from model output."""
    text = text.replace(",", "")  # remove commas in numbers
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return numbers[-1] if numbers else None

def reward_correct_answer(completions, answer, **kwargs):
    """
    +1.0  if final number matches ground truth
     0.0  if wrong or no number found
    """
    rewards = []
    for completion in completions:
        predicted = extract_number(completion)
        expected  = extract_number(answer[0])
        if predicted and expected and predicted == expected:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

def reward_step_by_step(completions, **kwargs):
    """
    Bonus reward for showing reasoning steps.
    Encourages the model to think before answering.
    """
    rewards = []
    for completion in completions:
        score = 0.0
        if len(completion.split()) > 20:   score += 0.1  # not too short
        if any(w in completion.lower() for w in
               ["step", "first", "then", "so", "therefore", "="]):
            score += 0.1                                  # uses reasoning words
        if re.search(r'\d+', completion):  score += 0.1  # contains numbers
        rewards.append(score)
    return rewards

def combined_reward(completions, answer, **kwargs):
    """Final reward = correctness + reasoning bonus."""
    correct  = reward_correct_answer(completions, answer, **kwargs)
    stepwise = reward_step_by_step(completions, **kwargs)
    return [c + s for c, s in zip(correct, stepwise)]

# print("✅ Reward functions defined!")
# print("   • reward_correct_answer  → +1.0 for correct final number")
# print("   • reward_step_by_step    → +0.3 for showing reasoning")
# print("   • combined_reward        → both combined (max 1.3)")

## 💾 Step 7: Checkpoint Callback

In [ ]:
from transformers import TrainerCallback

class CheckpointCallback(TrainerCallback):
    """Saves adapter + tokenizer after every N epochs."""

    def __init__(self, save_dir, save_every_n_epochs=1):
        self.save_dir   = save_dir
        self.save_every = save_every_n_epochs
        self._tokenizer = None

    def on_train_begin(self, args, state, control, model=None, **kwargs):
        self._tokenizer = tokenizer

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(state.epoch)
        if epoch % self.save_every == 0:
            path = os.path.join(self.save_dir, f'checkpoint-epoch-{epoch}')
            os.makedirs(path, exist_ok=True)
            model.save_pretrained(path)
            if self._tokenizer:
                self._tokenizer.save_pretrained(path)
            print(f'\n💾 Checkpoint saved → {path}')

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(
                f"   Step {state.global_step:>5} | epoch {state.epoch:.2f} | "
                f"loss {logs.get('loss', 'N/A')} | "
                f"reward {logs.get('reward', 'N/A')} | "
                f"reward_std {logs.get('reward_std', 'N/A')} | "
                f"kl {logs.get('kl', 'N/A')}"
            )

print('✅ Callback ready!')

✅ Callback ready!


## 🏋️ Step 8: Train with GRPO

In [ ]:
# Custom callback to evaluate accuracy after each epoch
from transformers import TrainerCallback

class AccuracyEvalCallback(TrainerCallback):
    def __init__(self, model, tokenizer, dataset, num_samples=50):
        self.model = model
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.num_samples = num_samples
    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\nEvaluating accuracy after epoch {int(state.epoch)}...")
        evaluate_model_accuracy(self.model, self.tokenizer, self.dataset, self.num_samples)

# Add to callbacks in trainer
# callbacks = [CheckpointCallback(...), AccuracyEvalCallback(model, tokenizer, dataset, 50)]


In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Add AccuracyEvalCallback to callbacks
callbacks = [
    CheckpointCallback(save_dir=SAVE_DIR, save_every_n_epochs=1),
    AccuracyEvalCallback(model, tokenizer, dataset, 50)
]

grpo_config = GRPOConfig(
    output_dir                    = GRPO_OUTPUT_DIR,
    num_train_epochs              = GRPO_EPOCHS,
    per_device_train_batch_size   = GRPO_BATCH_SIZE,
    gradient_accumulation_steps   = GRPO_GRAD_ACCUM,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    learning_rate                 = GRPO_LR,
    lr_scheduler_type             = 'cosine',
    warmup_ratio                  = 0.1,
    fp16                          = torch.cuda.is_available(),
    logging_strategy              = 'steps',
    logging_steps=50,
    save_strategy                 = 'steps',
    save_steps                    = 200,
    save_total_limit              = 3,
    report_to                     = 'none',
    num_generations               = NUM_GENERATIONS,
    max_prompt_length             = 256,
    max_completion_length         = MAX_NEW_TOKENS,
    beta                          = KL_BETA,
)

trainer = GRPOTrainer(
    model            = model,
    args             = grpo_config,
    reward_funcs     = combined_reward,
    train_dataset    = dataset,
    processing_class = tokenizer,
    callbacks        = callbacks,
)

print('🚀 Starting GRPO training...')
print(f'   Samples:     {len(dataset)}')
print(f'   Epochs:      {GRPO_EPOCHS}')
print(f'   LR:          {GRPO_LR}')
print(f'   Generations: {NUM_GENERATIONS} per prompt')
print(f'   KL beta:     {KL_BETA}')
print(f'   Save dir:    {SAVE_DIR}')
print('=' * 55)

result = trainer.train()


🚀 Starting GRPO training...
   Samples:     3000
   Epochs:      1
   LR:          5e-06
   Generations: 2 per prompt
   KL beta:     0.04
   Save dir:    /content/drive/MyDrive/RL Assignement/grpo_checkpoints


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss


## 💾 Step 9: Save Final Model

In [ ]:
final_path = os.path.join(SAVE_DIR, 'grpo_final')
os.makedirs(final_path, exist_ok=True)

model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)

summary = {
    'base_model':      MODEL_ID,
    'sft_adapter':     ADAPTER_PATH,
    'dataset':         'openai/gsm8k',
    'grpo_samples':    GRPO_SAMPLES,
    'epochs':          GRPO_EPOCHS,
    'lr':              GRPO_LR,
    'num_generations': NUM_GENERATIONS,
    'beta':            KL_BETA,
    'final_loss':      result.training_loss,
    'runtime_min':     result.metrics['train_runtime'] / 60,
}
with open(os.path.join(SAVE_DIR, 'grpo_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f'✅ GRPO training complete!')
print(f'   Train loss:  {result.training_loss:.4f}')
print(f'   Runtime:     {result.metrics["train_runtime"] / 60:.1f} min')
print(f'   Final model → {final_path}')
print(f'   Summary     → {SAVE_DIR}/grpo_summary.json')
print(f'\n📂 Output structure:')
print(f'''
{SAVE_DIR}/
├── checkpoint-epoch-1/
├── checkpoint-epoch-2/
├── checkpoint-epoch-3/
├── grpo_final/          ← load this for inference
└── grpo_summary.json
''')